# 72 Job · Tarea 03 · For Each por Género

Este notebook es la tarea anidada de un `For each`: Lakeflow Jobs lo ejecuta N veces y sustituye `{{input}}` por cada elemento de la lista. Vas a calcular y persistir una métrica independiente por género. El objetivo DCEA es **implementar control flows de looping con Lakeflow Jobs**.

In [ ]:
CATALOG = "big_data_ii_2025"
SCHEMA  = "spark_examples"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Trabajando en {CATALOG}.{SCHEMA}")

In [ ]:
dbutils.widgets.text("genero", "Drama", "Género de MovieLens")
genero = dbutils.widgets.get("genero").strip()
assert genero, "El parámetro genero no puede quedar vacío."

TABLA_SILVER = f"{CATALOG}.{SCHEMA}.movielens_silver"
assert spark.catalog.tableExists(TABLA_SILVER), (
    f"No existe {TABLA_SILVER}. Ejecutá primero la tarea 02_control_calidad "
    "o el notebook 71_Job_02_Control_Calidad."
)

In [ ]:
from pyspark.sql import functions as F

RUTA_VOLUMEN = f"/Volumes/{CATALOG}/{SCHEMA}/movielens/ml-latest-small"
RUTA_MOVIES = f"{RUTA_VOLUMEN}/movies.csv"
TABLA_MOVIES_FALLBACK = f"{CATALOG}.{SCHEMA}.movielens_movies_raw"

def existe_archivo(ruta):
    try:
        dbutils.fs.ls(ruta)
        return True
    except Exception:
        return False

if existe_archivo(RUTA_MOVIES):
    peliculas = (spark.read.option("header", True).option("inferSchema", True)
        .csv(RUTA_MOVIES))
elif spark.catalog.tableExists(TABLA_MOVIES_FALLBACK):
    peliculas = spark.table(TABLA_MOVIES_FALLBACK)
else:
    raise FileNotFoundError(
        f"No se encontró la dimensión de películas de MovieLens.\n"
        f"Se buscó en:\n"
        f"  1. Volumen : {RUTA_MOVIES}\n"
        f"  2. Tabla   : {TABLA_MOVIES_FALLBACK}\n"
        f"Cargá el dataset antes de continuar (notebook de ingesta de MovieLens) "
        f"o corregí la ruta del volumen en la celda de configuración."
    )

ratings_genero = (spark.table(TABLA_SILVER).alias("r")
    .join(peliculas.alias("m"), F.col("r.movieId") == F.col("m.movieId"), "inner")
    .filter(F.array_contains(F.split(F.col("m.genres"), r"\|"), genero)))

metrica = ratings_genero.agg(
    F.lit(genero).alias("genero"),
    F.count("r.rating").cast("long").alias("cantidad_ratings"),
    F.round(F.avg("r.rating"), 4).alias("rating_promedio"),
    F.round(F.stddev_samp("r.rating"), 4).alias("desviacion_estandar"),
    F.countDistinct("r.movieId").cast("long").alias("peliculas_distintas"),
    F.current_timestamp().alias("actualizado_ts"))
display(metrica)

In [ ]:
TABLA_METRICAS = f"{CATALOG}.{SCHEMA}.movielens_metricas_genero"
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLA_METRICAS} (
  genero STRING,
  cantidad_ratings BIGINT,
  rating_promedio DOUBLE,
  desviacion_estandar DOUBLE,
  peliculas_distintas BIGINT,
  actualizado_ts TIMESTAMP
) USING DELTA
""")

metrica.createOrReplaceTempView("_metrica_genero_actual")
spark.sql(f"""
MERGE INTO {TABLA_METRICAS} AS destino
USING _metrica_genero_actual AS origen
ON destino.genero = origen.genero
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

try:
    dbutils.jobs.taskValues.set(key="genero_procesado", value=genero)
except Exception as e:
    print(f"Ejecución fuera de un job: no se publicó el task value ({e})")
print(f"genero_procesado = {genero}")

## Cierre

- Recibiste el elemento actual del `For each` mediante el widget `genero`.
- Uniste ratings Silver con la dimensión de películas.
- Calculaste cuatro métricas analíticas para un género.
- Persististe el resultado con `MERGE`, de forma segura para iteraciones paralelas.